# Phase 2 — Final Metrics: AUC, Bootstrap CIs, Confusion Matrices, Per-Class Stats

Reloads each saved model and computes the metrics the paper needs but the original runs did not all capture:
- **Multi-class AUC-ROC** (one-vs-rest) from softmax probabilities
- **Bootstrap 95% confidence intervals** for accuracy, MCC, G-mean, AUC (resampling the 29 test patients)
- **Confusion-matrix counts** and **per-class precision/recall/F1** (saved as data for you to plot manually)
- A separate **5-fold CV** estimate for the headline condition C3b

All results saved to Drive as JSON/CSV. No retraining except the C3b CV. Confusion matrices and infographics are created manually by you from the saved counts.

> Set Runtime → GPU. Reloading frozen-encoder models + test inference is fast.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!pip install -q "monai==1.5.0" nibabel torchio imbalanced-learn scikit-learn
import os; os._exit(0)


## 1. Verify env

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import torch, numpy, monai
print('NumPy',numpy.__version__,'MONAI',monai.__version__)
from monai.networks.nets import SwinUNETR
assert torch.cuda.is_available()


## 2. Config & shared data

In [ ]:
import os, json, numpy as np, torch, datetime
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchio as tio, pandas as pd
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix, roc_auc_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from monai.networks.nets import SwinUNETR
device=torch.device('cuda')

DRIVE='/content/drive/My Drive/ADNI_NewDS/'; RESULTS=os.path.join(DRIVE,'results')
BRAIN_DIR=os.path.join(RESULTS,'processed_mri_scans_brain')
OLD_DIR=os.path.join(RESULTS,'processed_mri_scans_swin')
SPLITS=os.path.join(RESULTS,'patient_id_splits.npz')
CSV=os.path.join(RESULTS,'project_data_cleaned.csv'); BIO=os.path.join(RESULTS,'preprocessed_biomarker_sequences.npy')
PHASE2=os.path.join(DRIVE,'Phase2_Collapse_Study'); RES=os.path.join(PHASE2,'results')
SSL_DIR=os.path.join(PHASE2,'ssl_pretrain')
METRICS_DIR=os.path.join(PHASE2,'final_metrics'); os.makedirs(METRICS_DIR, exist_ok=True)
IMG_SIZE=(96,96,96); FEATURE_SIZE=48; VIT_DIM=768; NUM_CLASSES=3; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)

sp=np.load(SPLITS,allow_pickle=True)
pids_tr,pids_va,pids_te=list(sp['pids_train']),list(sp['pids_val']),list(sp['pids_test'])
y_tr,y_va,y_te=list(sp['labels_train']),list(sp['labels_val']),list(sp['labels_test'])
clin_df=pd.read_csv(CSV)
FEATS=['AGE','PTGENDER','PTEDUCAT','APOE4','MMSE','ADAS13','RAVLT_immediate','RAVLT_learning','RAVLT_forgetting','FAQ']
patient_clin={p:torch.tensor(g[FEATS].values,dtype=torch.float32) for p,g in clin_df.groupby('PTID')}
bio_raw=np.load(BIO,allow_pickle=True); bio_obj=bio_raw.item() if (hasattr(bio_raw,'dtype') and bio_raw.dtype==object and bio_raw.ndim==0) else bio_raw
patient_bio={k:torch.tensor(np.asarray(v),dtype=torch.float32) for k,v in bio_obj.items()}
CLIN_DIM=len(FEATS); BIO_DIM=next(iter(patient_bio.values())).shape[-1]
print('data loaded. test n =', len(pids_te))


## 3. Metric helpers (AUC, G-mean, bootstrap CIs)

In [ ]:
def gmean_from_cm(cm):
    g=[]
    for i in range(cm.shape[0]):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-(tp+fn+fp)
        se=tp/(tp+fn) if tp+fn>0 else 0; sp=tn/(tn+fp) if tn+fp>0 else 0; g.append((se*sp)**0.5)
    return float(np.mean(g))

def multiclass_auc(y_true, probs):
    try:
        return float(roc_auc_score(y_true, probs, multi_class='ovr', average='macro'))
    except Exception as e:
        return None

def all_metrics(y_true, y_pred, probs):
    y_true=np.array(y_true); y_pred=np.array(y_pred)
    cm=confusion_matrix(y_true,y_pred,labels=[0,1,2])
    return {'acc':float(accuracy_score(y_true,y_pred)),'mcc':float(matthews_corrcoef(y_true,y_pred)),
            'gmean':gmean_from_cm(cm),'auc':multiclass_auc(y_true,probs) if probs is not None else None}

def bootstrap_ci(y_true, y_pred, probs, n_boot=2000, seed=42):
    rng=np.random.default_rng(seed); y_true=np.array(y_true); y_pred=np.array(y_pred)
    probs=np.array(probs) if probs is not None else None
    n=len(y_true); accs=[];mccs=[];gms=[];aucs=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n)
        if len(np.unique(y_true[idx]))<2: continue
        accs.append(accuracy_score(y_true[idx],y_pred[idx]))
        mccs.append(matthews_corrcoef(y_true[idx],y_pred[idx]))
        gms.append(gmean_from_cm(confusion_matrix(y_true[idx],y_pred[idx],labels=[0,1,2])))
        if probs is not None:
            a=multiclass_auc(y_true[idx],probs[idx])
            if a is not None: aucs.append(a)
    def ci(v): return [float(np.percentile(v,2.5)),float(np.percentile(v,97.5))] if v else None
    return {'acc_ci':ci(accs),'mcc_ci':ci(mccs),'gmean_ci':ci(gms),'auc_ci':ci(aucs)}

def per_class(y_true,y_pred):
    p,r,f,s=precision_recall_fscore_support(y_true,y_pred,labels=[0,1,2],zero_division=0)
    return {c:{'precision':float(p[i]),'recall':float(r[i]),'f1':float(f[i]),'support':int(s[i])} for i,c in enumerate(['CN','MCI','Dementia'])}
print('metric helpers ready')


## 4. Encoder + model classes (standard + improved gate)

In [ ]:
class SwinEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.swin=SwinUNETR(in_channels=1,out_channels=1,feature_size=FEATURE_SIZE,use_checkpoint=True).swinViT; self.pool=nn.AdaptiveAvgPool3d(1)
    def forward(self,x): return self.pool(self.swin(x)[-1]).flatten(1)

class LSTMBranch(nn.Module):
    def __init__(s,ind,h=128,o=128):
        super().__init__(); s.l=nn.LSTM(ind,h,2,batch_first=True,dropout=0.2); s.f=nn.Linear(h,o); s.r=nn.ReLU()
    def forward(s,x): _,(h,_)=s.l(x); return s.r(s.f(h[-1]))

# Standard model (C2, C2b, C3b, C3c, C4-*): proj + dual LSTM + simple softmax gate
class TriModalStd(nn.Module):
    def __init__(s,enc,cd,bd,nc=3,unfreeze=False):
        super().__init__(); s.encoder=enc; s.unfreeze=unfreeze
        for p in s.encoder.parameters(): p.requires_grad=unfreeze
        s.img_proj=nn.Sequential(nn.Linear(VIT_DIM,256),nn.ReLU(),nn.Linear(256,128))
        s.clin_branch=LSTMBranch(cd); s.bio_branch=LSTMBranch(bd)
        s.fusion_gate=nn.Sequential(nn.Linear(384,128),nn.ReLU(),nn.Linear(128,3),nn.Softmax(dim=1))
        s.classifier=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.5),nn.Linear(64,nc))
    def imgfeat(s,m):
        ctx=torch.enable_grad() if s.unfreeze else torch.no_grad()
        with ctx: f=s.encoder(m.unsqueeze(1))
        return s.img_proj(f)
    def forward(s,m,c,b):
        fi=s.imgfeat(m); fc=s.clin_branch(c); fb=s.bio_branch(b)
        st=torch.stack([fi,fc,fb],1); w=s.fusion_gate(torch.cat([fi,fc,fb],1))
        return s.classifier((st*w.unsqueeze(-1)).sum(1))
print('model classes ready (extend with improved/forced gate if reloading C5/C6)')


## 5. Build cached test features once, define eval

In [ ]:
etf=tio.Compose([tio.Resize(IMG_SIZE), tio.ZNormalization(masking_method=tio.ZNormalization.mean)])
class DS(Dataset):
    def __init__(s,pids,labels,mri_dir): s.pids=list(pids); s.y=torch.tensor(np.array(labels),dtype=torch.long); s.dir=mri_dir
    def __len__(s): return len(s.pids)
    def __getitem__(s,i):
        p=s.pids[i]; vol=np.load(os.path.join(s.dir,f'{p}_processed.npy'))
        subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
        return etf(subj).mri.tensor.squeeze(0), patient_clin[p], patient_bio[p], s.y[i]
def coll(b):
    m,c,bi,y=zip(*b); return torch.stack(m),pad_sequence(c,batch_first=True),pad_sequence(bi,batch_first=True),torch.stack(y)

@torch.no_grad()
def eval_model(model, mri_dir):
    model.eval(); te=DataLoader(DS(pids_te,y_te,mri_dir),batch_size=8,shuffle=False,collate_fn=coll)
    P=[];T=[];PR=[]
    for m,c,b,y in te:
        logits=model(m.to(device),c.to(device),b.to(device))
        pr=F.softmax(logits,dim=1).cpu().numpy()
        P+=logits.argmax(1).cpu().tolist(); T+=y.tolist(); PR.append(pr)
    PR=np.concatenate(PR)
    m=all_metrics(T,P,PR); m.update(bootstrap_ci(T,P,PR)); 
    return {'metrics':m,'per_class':per_class(T,P),'confusion_matrix':confusion_matrix(T,P,labels=[0,1,2]).tolist(),'y_true':list(map(int,T)),'y_pred':list(map(int,P))}
print('eval ready')


## 6. Reload each model, compute & save full metrics
Set the (condition → weights path, encoder source, preprocessing dir, model class) map. C5/C6 need their gate classes added.

In [ ]:
# MONAI CT-pretrained weights (for conditions that used them as init before loading saved state)
def fresh_ct_encoder():
    return SwinEncoder()  # state will be overwritten by the saved best_model.pth anyway

# condition -> (weights_subdir, mri_dir, builder)
def build_std(unfreeze=False):
    return TriModalStd(SwinEncoder(),CLIN_DIM,BIO_DIM,NUM_CLASSES,unfreeze=unfreeze)

CONDITIONS={
  'condition3b_brain_preprocessed_frozen': (BRAIN_DIR, lambda: build_std(False)),
  'condition3c_brain_finetuned':            (BRAIN_DIR, lambda: build_std(True)),
  'condition4_ssl_vicreg_frozen':           (BRAIN_DIR, lambda: build_std(False)),
  'condition4_ssl_mae_frozen':              (BRAIN_DIR, lambda: build_std(False)),
  'condition2_monai_pretrained_frozen':     (OLD_DIR,   lambda: build_std(False)),
}
# NOTE: C5/C6 use improved/forced gate classes — add them here if you want those too.

results={}
for cond,(mri_dir,builder) in CONDITIONS.items():
    wpath=os.path.join(RES,cond,'best_model.pth')
    if not os.path.exists(wpath): print('SKIP (no weights):',cond); continue
    model=builder().to(device)
    sd=torch.load(wpath,map_location='cpu')
    missing,unexpected=model.load_state_dict(sd,strict=False)
    frac=1-len(missing)/len(model.state_dict())
    print(f'{cond}: loaded {frac:.0%} keys', '(OK)' if frac>0.9 else '(CHECK)')
    try:
        results[cond]=eval_model(model,mri_dir)
        m=results[cond]['metrics']
        print(f"   acc={m['acc']:.3f} mcc={m['mcc']:.3f} gmean={m['gmean']:.3f} auc={m['auc']}")
        print(f"   95% CI acc {m['acc_ci']}  auc {m['auc_ci']}")
    except Exception as e:
        print('   ERROR evaluating:',e)
json.dump(results, open(os.path.join(METRICS_DIR,'final_metrics_all.json'),'w'), indent=2)
print('\nSaved final_metrics_all.json')


## 7. 5-fold CV for the headline condition (C3b) with all metrics
Retrains C3b's frozen-encoder recipe across folds to give a CV mean±std for acc/MCC/G-mean/AUC — the rigorous estimate for the main result.

In [ ]:
# Cache C3b image features once (frozen CT encoder loaded from C3b weights' encoder part)
c3b_w=os.path.join(RES,'condition3b_brain_preprocessed_frozen','best_model.pth')
base=build_std(False).to(device); base.load_state_dict(torch.load(c3b_w,map_location='cpu'),strict=False); base.eval()

pool=pids_tr+pids_va; pooly=np.array(y_tr+y_va)
@torch.no_grad()
def feat(pid,mri_dir=BRAIN_DIR):
    vol=np.load(os.path.join(mri_dir,f'{pid}_processed.npy'))
    subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
    t=etf(subj).mri.tensor.unsqueeze(0).to(device)
    return base.imgfeat(t.squeeze(1) if t.dim()==5 else t).cpu().squeeze(0) if False else base.img_proj(base.encoder(t)).cpu().squeeze(0)
# simpler: cache raw encoder feature then reuse proj inside fold models is complex; 
# instead retrain full frozen model per fold (fast since encoder frozen)

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED); pa=np.array(pool)
fold_metrics=[]
for fi,(tri,vai) in enumerate(skf.split(pa,pooly)):
    tp,vp=pa[tri].tolist(),pa[vai].tolist(); ty,vy=pooly[tri],pooly[vai]
    # fresh frozen model initialised from C3b encoder weights
    mdl=build_std(False).to(device)
    enc_sd={k:v for k,v in torch.load(c3b_w,map_location='cpu').items() if k.startswith('encoder.')}
    mdl.load_state_dict(enc_sd,strict=False)
    trL=DataLoader(DS(tp,ty,BRAIN_DIR),batch_size=8,shuffle=True,collate_fn=coll)
    vaL=DataLoader(DS(vp,vy,BRAIN_DIR),batch_size=8,shuffle=False,collate_fn=coll)
    cw=compute_class_weight('balanced',classes=np.unique(ty),y=ty)
    crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float).to(device))
    opt=torch.optim.AdamW([p for p in mdl.parameters() if p.requires_grad],lr=1e-4,weight_decay=1e-5)
    best=-1;bs=None;wait=0
    for ep in range(40):
        mdl.train()
        for m,c,b,y in trL:
            m,c,b,y=m.to(device),c.to(device),b.to(device),y.to(device)
            opt.zero_grad(); crit(mdl(m,c,b),y).backward(); opt.step()
        mdl.eval();P=[];T=[]
        with torch.no_grad():
            for m,c,b,y in vaL: P+=mdl(m.to(device),c.to(device),b.to(device)).argmax(1).cpu().tolist();T+=y.tolist()
        vm=matthews_corrcoef(T,P)
        if vm>best: best=vm;bs={k:v.cpu().clone() for k,v in mdl.state_dict().items()};wait=0
        else:
            wait+=1
            if wait>=10: break
    if bs: mdl.load_state_dict(bs)
    # eval fold with probabilities
    mdl.eval();P=[];T=[];PR=[]
    with torch.no_grad():
        for m,c,b,y in vaL:
            lo=mdl(m.to(device),c.to(device),b.to(device)); PR.append(F.softmax(lo,1).cpu().numpy())
            P+=lo.argmax(1).cpu().tolist();T+=y.tolist()
    PR=np.concatenate(PR); fm=all_metrics(T,P,PR); fold_metrics.append(fm)
    print(f"fold{fi+1}: acc={fm['acc']:.3f} mcc={fm['mcc']:.3f} gmean={fm['gmean']:.3f} auc={fm['auc']}")

def ms(key): 
    v=[f[key] for f in fold_metrics if f[key] is not None]; import numpy as np; return [float(np.mean(v)),float(np.std(v))]
cv_summary={k:ms(k) for k in ['acc','mcc','gmean','auc']}
print('\nC3b 5-fold CV (mean, std):',cv_summary)
json.dump({'folds':fold_metrics,'summary':cv_summary}, open(os.path.join(METRICS_DIR,'c3b_cv_metrics.json'),'w'), indent=2)
print('Saved c3b_cv_metrics.json. Paste output back to Claude.')
